# Aether Paper Artifact: Clean Kaggle Start

In Kaggle, enable a GPU and Internet, then attach datasets through **Add Input**. Run every code cell in order. This notebook clones the published Aether revision, prepares the Python and Java runtime, and runs correctness smoke tests. It does not run a performance campaign until real dataset manifests are configured.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import time

REPOSITORY = "https://github.com/Grgur00/Aether-Engine.git"
GIT_REF = "main"  # Replace with a commit SHA before a confirmatory campaign.
REPO = Path("/kaggle/working/Aether-Engine")
RESULTS = Path("/kaggle/working/aether-results")
PY = Path("/kaggle/working/aether-paper-venv/bin/python")

# Set these only after attaching a dataset with real source images and masks.
OCT_FULL_MANIFEST = None
COCO_IMAGE_ROOT = None
COCO_ANNOTATIONS = None
IMAGENET_IMAGE_ROOT = None
DALI_DATASETS_CONFIG = None

RUN_OCT_PILOT = False
RUN_DALI_SMOKE = True
RUN_DALI_COMPARISON = False

In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO)], check=True)
if GIT_REF != "main":
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "origin/main"], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Aether commit:", commit)

In [ ]:
subprocess.run(["bash", str(REPO / "kaggle/setup.sh")], cwd=REPO, check=True)

runtime = json.loads(Path("/kaggle/working/aether-paper-runtime.json").read_text())
os.environ["JAVA_HOME"] = runtime["javaHome"]
os.environ["PATH"] = runtime["javaHome"] + "/bin:" + os.environ["PATH"]
os.environ["PYTHONPATH"] = f"{REPO / 'clients/python'}:{REPO / 'scripts'}"

if not PY.is_file():
    raise RuntimeError(f"Kaggle environment was not created: {PY}")
classpath = REPO / "modules/aether-training-cache/build/paper-runtime-classpath.txt"
if not classpath.is_file() or not classpath.read_text().strip():
    raise RuntimeError("Java runtime classpath was not generated by setup")
print("Ready:", PY)
print("Java classpath:", classpath)

In [ ]:
def run(script, *arguments):
    command = [str(PY), str(REPO / "scripts" / script), *map(str, arguments)]
    print(" ".join(command))
    subprocess.run(command, cwd=REPO, check=True)

run("validate_gpu.py")
subprocess.run([str(PY), "-m", "pip", "install", "-r", str(REPO / "env/requirements-dali.lock")], check=True)
run("validate_gpu.py", "--require-dali")

## Correctness Smoke Tests

These use generated fixtures and validate the Java engine, mmap baseline, RAW/RAM parity, DALI GPU tensors, and final model parity. They are correctness evidence only, not paper performance measurements.

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
SMOKE_OUTPUT = RESULTS / f"smoke-{stamp}"
DALI_SMOKE_OUTPUT = RESULTS / f"dali-smoke-{stamp}"

run("artifact_smoke.py", "--output", SMOKE_OUTPUT)
if RUN_DALI_SMOKE:
    run("dali_smoke.py", "--output", DALI_SMOKE_OUTPUT)

print("Smoke outputs:", SMOKE_OUTPUT, DALI_SMOKE_OUTPUT if RUN_DALI_SMOKE else "DALI disabled")

## Real Data and Pilot

Set `OCT_FULL_MANIFEST` only when it points to a persisted manifest containing real OCT source images and semantic masks. The annotation-only input previously inspected is not valid for this step. This cell creates the fixed V1-to-V2 experiment and runs the 10-block pilot only when `RUN_OCT_PILOT` is set to `True`.

In [ ]:
if RUN_OCT_PILOT:
    if OCT_FULL_MANIFEST is None:
        raise RuntimeError("Set OCT_FULL_MANIFEST to a real image-and-mask manifest before running the OCT pilot")
    full_manifest = Path(OCT_FULL_MANIFEST)
    if not full_manifest.is_file():
        raise FileNotFoundError(full_manifest)

    evolution = Path("/kaggle/working/aether-data/oct5k-evolution")
    if evolution.exists():
        shutil.rmtree(evolution)
    run("prepare_evolution.py", "--manifest", full_manifest, "--output", evolution,
        "--v1-size", 1170, "--v2-size", 1505, "--reusable", 1003, "--seed", 20260904)

    config = json.loads((REPO / "configs/paper/datasets.json").read_text())
    config["oct5k"]["manifestV1"] = str(evolution / "v1.csv")
    config["oct5k"]["manifestV2"] = str(evolution / "v2.csv")
    config_path = Path("/kaggle/working/aether-datasets.json")
    config_path.write_text(json.dumps(config, indent=2))
    run("validate_manifests.py", "--config", config_path, "--datasets", "oct5k")
    run("run_matrix.py", "--config", config_path, "--datasets", "oct5k", "--repeats", 10,
        "--resume", "--output", RESULTS / "oct5k-pilot")
    run("analyze.py", "--input", RESULTS / "oct5k-pilot", "--pilot", "--output", RESULTS / "oct5k-pilot-analysis")

## Optional DALI Dataset Campaign and Evidence

Set both COCO and ImageNet source paths before enabling this separate 12-block comparison. The generated DALI smoke above is not included in this campaign.

In [ ]:
if RUN_DALI_COMPARISON:
    if DALI_DATASETS_CONFIG is None:
        raise RuntimeError("Set DALI_DATASETS_CONFIG to a validated COCO/ImageNet manifest config")
    dali_config = Path(DALI_DATASETS_CONFIG)
    if not dali_config.is_file():
        raise FileNotFoundError(dali_config)
    run("validate_manifests.py", "--config", dali_config, "--datasets", "coco,imagenet")
    run("dali_comparison.py", "--config", dali_config, "--datasets", "coco,imagenet",
        "--repeats", 12, "--resume", "--output", RESULTS / "dali-comparison")

## Save Evidence
Run this cell to create one verified `/kaggle/working/aether-results-only.zip` containing reports, logs, runtime metadata and SHA256SUMS. You can also run it after an earlier cell fails to save partial results.


In [ ]:
# Export all reports, logs and runtime metadata in one verified ZIP.
import sys
sys.path.insert(0, str(REPO / 'scripts'))
from kaggle_results import bundle_results
archive = bundle_results(RESULTS, '/kaggle/working/aether-results-only.zip',
                         '/kaggle/working/aether-paper-runtime.json')
print('Download:', archive)
